In [55]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import scipy.stats as stats

mV = 1/1000.
Hz = 1
kHz = 1000
seconds = 1
minutes = 60*seconds
V = 1

In [61]:
class Sensor:

    def __init__(self, 
                 white_noise_V : float,
                 amp_noise_V : float,
                 f_noise_Hz : float,
                 sens_termopar : float,
                ):
        self.white_noise_V = white_noise_V
        self.amp_noise_V = amp_noise_V
        self.f_noise_Hz = f_noise_Hz
        self.sens_termopar = sens_termopar

    def run(self, 
            t : np.array,
            T : float, 
           ):
        # real temperature value
        v = T*self.sens_termopar
        # noise injection
        white_noise = np.random.normal(0, self.white_noise_V, len(t))
        network_noise = self.amp_noise_V * np.sin( 2*np.pi*self.f_noise_Hz*t)
        v_out = v + white_noise + network_noise
        return t, v_out

class AmpOp:
    def __init__(self, 
                 A_V : float = 100, 
                 v_offset : float = 15*mV, 
                 v_min: float = 0*V, 
                 v_max: float = 5*V 
                ):
        self.A_V = A_V
        self.v_offset = v_offset
        self.v_min = v_min
        self.v_max = v_max

    def run(self, v_in ):
        # amplification
        v_out = v_in * self.A_V + self.v_offset
        v_out = np.clip (v_out , self.v_min, self.v_max)
        return v_out

class ADC:

    def __init__(self, 
                 n_bits : int,
                 fs : float, 
                 fs_real_world : float,
                 v_min : float = 0*V,
                 v_max : float = 3.3*V
                ):
        self.n_bits = n_bits
        self.v_min = v_min
        self.v_max = v_max
        self.fs = fs
        self._step = 1 if int(fs_real_world/fs) < 1 else int(fs_real_world/fs)
    
    def run(self, t, v_in ):

        n = t[::self._step]
        v_n_in = v_in[::self._step]
        steps  = 2**self.n_bits
        quantum = (self.v_max - self.v_min) / steps
        v_n_in = np.clip(v_n_in, self.v_min, self.v_max)
        word = np.floor((v_n_in - self.v_min) / quantum).astype(int)
        word = np.clip(word, 0, steps - 1)
        v_out_digit = self.v_min + word * quantum + (quantum / 2)
        return n, v_out_digit

class Filter:
    def __init__(self, fs : float, f_cut : float, order : int = 4):
        self.fs = fs
        self.f_cut = f_cut
        self.order = order

    def run(self, v_in ):
        nyquist = 0.5 * self.fs
        f_cut_norm = self.f_cut / nyquist
        b, a = signal.butter(self.order, f_cut_norm, btype='low')
        return signal.lfilter(b, a, v_in)
        

class Temperature:
    def __init__( self,
                  sens_termopar : float,
                  A_V : float = 100,
                  v_offset : float = 15*mV
                ):
        self.sens_termopar = sens_termopar
        self.A_V = A_V
        self.v_offset = v_offset
                  
    def run(self, v_in ):
        T = v_in - self.v_offset
        T = T / (self.A_V * self.sens_termopar)
        return T



In [ ]:
# model
def normal(x, mean, amplitude, standard_deviation):
    return amplitude * np.exp( - (x - mean)**2 / (2*standard_deviation ** 2))



class ChiSquare:

    def __init__(self, mean_window : int = 200, num_bins : int = 50 , xmin : float, xmax : float, model = model):
        self.model=model
        self.xmin = xmin
        self.xmax = xmax
        self.num_bins = num_bins
        self.mean_window = mean_window


    def get_density( data , xbins : int, xmin : float, xmax : float):
        freq_obs, bin_edges = np.histogram(data, bins=nbins, range=(xmin,xmax), density=False)
        tot_obs             = len(data)
        bin_sizes           = np.diff(bin_edges)
        density_obs         = freq_obs/(np.sum(freq_obs)*bin_sizes)
        return density_obs, freq_obs, bin_edges
    

    def run(self, T):

        
        degrees_of_freedon = self.num_bins - 1

        T_window = T[-self.mean_window:]
        T_mean = np.mean(T_window)
        T_residual = T_window - T_mean
        T_residual_std = np.std(T_residual)
        

        density_obs, freq_obs, bin_edges = get_density(T_residual, nbins=self.num_bins, xmin=self.xmin, xmax=self.xmax)
        bin_centers      = bin_edges[:-1] + np.diff(bin_edges) / 2 
        bin_sizes        = np.diff(bin_edges) 
        prob_obs         = (density_obs*bin_sizes)


        params, _          = curve_fit(model, bin_centers, density_obs )
        x                  = np.linspace(bin_edges[0], bin_edges[-1], 10000)
        density_expected   = model(bin_centers, *params)

        density_expected_for_each_bin_center = model(bin_centers, *params) 
        prob_expected      = density_expected_for_each_bin_center * bin_sizes 
        freq_expected      = (prob_expected*n).astype(int)
        
        chi2 = 0
        for k in range(nbins):
          if freq_expected[k] != 0:
            chi2+= ((freq_obs[k] - freq_expected[k])**2 / freq_expected[k])




    num_classes = 10
    frequencias_observadas, limites_classes = np.histogram(ruido_residual, bins=num_classes)

    frequencias_esperadas = []
    n_total = len(ruido_residual)
    media_ruido = np.mean(ruido_residual)

    for i in range(num_classes):
        p_classe = stats.norm.cdf(limites_classes[i+1], loc=media_ruido, scale=desvio_padrao_ruido) - \
                   stats.norm.cdf(limites_classes[i], loc=media_ruido, scale=desvio_padrao_ruido)
        frequencias_esperadas.append(p_classe * n_total)

    frequencias_esperadas = np.array(frequencias_esperadas)
    frequencias_esperadas = np.where(frequencias_esperadas == 0, 1e-4, frequencias_esperadas)

    graus_liberdade = num_classes - 1 - 2
    qui_quadrado_calc, p_valor = stats.chisquare(f_obs=frequencias_observadas, f_exp=frequencias_esperadas, ddof=2)
    qui_quadrado_critico = stats.chi2.ppf(1 - 0.05, graus_liberdade)

    status_teste = qui_quadrado_calc <= qui_quadrado_critico and not np.isnan(qui_quadrado_calc)

In [62]:

max_time_s = 1*seconds
fs_real_world = 5*kHz
t = np.arange(0, max_time_s, 1/fs_real_world)

sensor = Sensor(2*mV, 5*mV, 60*Hz, 29.1*mV/700, )
amp = AmpOp(100)
adc = ADC(10, 1*kHz, fs_real_world)

low_band = Filter( 1*kHz, 60*Hz , 5)

temp = Temperature ( 29.1*mV/700 )

t, v_out = sensor.run(t, 650)
v_out = amp.run(v_out) 
n , v_digit = adc.run( t, v_out)
v_digit_filtered = low_band.run(v_digit)
T_digit = temp.run(v_digit_filtered)